![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 02: AI Modelling Basics)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- Teaching content is licensed under CC BY 4.0 and code under MIT; see [LICENSING.md](../../LICENSING.md) for scope and exclusions.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 2A: Regression and ML Basics

<div align="center">

<table>
<thead>
<tr><th><strong>Item</strong></th><th><strong>Description</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Main output</td><td>A simple supervised regression pipeline with training, prediction and error analysis</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m02a-overview)
2. [Setup and Background](#m02a-setup)
3. [Core Concepts](#m02a-core-concepts)
4. [Guided Implementation](#m02a-guided-implementation)
5. [Testing and Analysis](#m02a-testing)
6. [Student Tasks](#m02a-student-tasks)
7. [Submission and Reflection](#m02a-submission)

---

<a id="m02a-overview"></a>

### 1. Overview and Learning Goals

This session introduces basic supervised machine learning through a small regression example. Regression is a useful starting point because the goal is easy to state: given one or more input features, predict a numeric output.

Later in the unit, you will work with embeddings, RAG systems, model evaluation, fine-tuning and agentic workflows. Those topics are more complex, but they still rely on the same modelling habits introduced here: define inputs and outputs, split data, fit a model, test the model and interpret errors.

In this notebook, we use a small synthetic dataset representing study time and quiz performance. The dataset is intentionally simple and inspectable. It is not meant to be a real education dataset. When future practicals require external data, use public unit materials first and then public datasets from [tulip-lab/open-data](https://github.com/tulip-lab/open-data).

This session also prepares you for [M02C-Embeddings-VectorData-Similarity](M02C-Embeddings-VectorData-Similarity.ipynb), where the input features are no longer simple numbers but vector representations of text. The modelling discipline is similar: represent data, compare outputs, evaluate behaviour and explain limitations.

By the end of this lab, you should be able to explain the difference between features and labels, split data into training and test sets, fit a simple linear regression model, interpret slope and intercept, calculate prediction error, and test normal, edge and failure behaviours in a small ML pipeline.

<a id="m02a-setup"></a>

### 2. Setup and Background

This notebook uses `numpy`, `pandas` and `matplotlib`, which are common tools for data handling and visualisation in Python. The model itself is implemented using a small closed-form linear regression calculation, rather than relying on a high-level machine learning library. This makes the mechanics visible.

The dataset uses two columns.

<div align="center">

<table>
<thead>
<tr><th><strong>Column</strong></th><th><strong>Role</strong></th><th><strong>Meaning</strong></th></tr>
</thead>
<tbody>
<tr><td align="left"><code>study_hours</code></td><td>Feature</td><td>Number of hours spent preparing for a quiz.</td></tr>
<tr><td align="left"><code>quiz_score</code></td><td>Label</td><td>Observed quiz score out of 100.</td></tr>
</tbody>
</table>

</div>

A feature is an input used by a model. A label is the target value that the model tries to predict. In a later practical, a feature could also be a text embedding, an image feature, a RAG retrieval score or an evaluation metric.

In [ ]:
import math
from typing import Any, Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Setup complete.")

In [ ]:
# A small synthetic dataset for teaching supervised regression.
# This is deliberately small and inspectable.
# For larger public examples in later sessions, prefer:
# https://github.com/tulip-lab/open-data

data = pd.DataFrame({
    "study_hours": [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
    "quiz_score":  [42,  48,  52,  57,  63,  68,  72,  78,  83,  88],
})

data

The table shows ten observations. Each row is one example. The model will learn a relationship between `study_hours` and `quiz_score`. Because the data is small, you can inspect every value and reason about the pattern directly.

You should notice that quiz scores generally increase as study hours increase. The relationship is not perfectly exact, but it is close to linear. This makes it suitable for a first regression example.

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(data["study_hours"], data["quiz_score"])
plt.xlabel("Study hours")
plt.ylabel("Quiz score")
plt.title("Study Hours vs Quiz Score")
plt.grid(True, alpha=0.3)
plt.show()

The scatter plot visualises the relationship. Each point is one observation. A regression model tries to find a line that summarises the trend. The line will not pass through every point exactly. Instead, it minimises the overall prediction error under a chosen criterion.

<a id="m02a-core-concepts"></a>

### 3. Core Concepts

Supervised learning means learning from examples where both the input and the target output are known. In this notebook, `study_hours` is the input feature and `quiz_score` is the known label.

For simple linear regression with one feature, the model has the form:

```text
prediction = slope × feature + intercept
```

The `slope` describes how much the prediction changes when the feature increases by one unit. The `intercept` describes the predicted value when the feature is zero. In this example, the slope roughly represents how many quiz-score points are associated with one additional hour of study.

A model should be evaluated on data that was not used to fit the model. This is why we use a training set and a test set. The training set is used to estimate the model parameters. The test set is used to estimate how well the model generalises to new examples.

<div align="center">

<table>
<thead>
<tr><th><strong>Term</strong></th><th><strong>Meaning</strong></th><th><strong>Why it matters</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Feature</td><td>Input value used by the model.</td><td>Defines what information the model can use.</td></tr>
<tr><td align="left">Label</td><td>Target value the model tries to predict.</td><td>Defines the learning objective.</td></tr>
<tr><td align="left">Training set</td><td>Data used to fit model parameters.</td><td>The model learns from this part.</td></tr>
<tr><td align="left">Test set</td><td>Data held out for evaluation.</td><td>Checks whether the model generalises.</td></tr>
<tr><td align="left">MAE</td><td>Mean absolute error.</td><td>Average absolute size of prediction mistakes.</td></tr>
</tbody>
</table>

</div>

In [ ]:
def train_test_split_simple(df: pd.DataFrame, test_size: int = 2) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Split a dataframe into training and test parts."""
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame.")

    if test_size <= 0:
        raise ValueError("test_size must be positive.")

    if test_size >= len(df):
        raise ValueError("test_size must be smaller than the dataset size.")

    train_df = df.iloc[:-test_size].copy()
    test_df = df.iloc[-test_size:].copy()
    return train_df, test_df


train_df, test_df = train_test_split_simple(data, test_size=2)

print("Training data:")
display(train_df)

print("Test data:")
display(test_df)

The output shows that the first eight rows are used for training and the last two rows are held out for testing. This split is intentionally simple. In a real modelling workflow, the split strategy should be chosen carefully. For example, time-ordered data should not be randomly shuffled if that would leak future information into training.

<a id="m02a-guided-implementation"></a>

### 4. Guided Implementation

We now implement simple linear regression using the closed-form least-squares solution for one input feature. The implementation is short, but the design is still structured: validate inputs, fit parameters, predict outputs and evaluate error.

The practical idea is that the model learns two parameters from training data and then uses those parameters to make predictions for new inputs.

In [ ]:
def fit_simple_linear_regression(x: Any, y: Any) -> Dict[str, Any]:
    """Fit a simple linear regression model y = slope * x + intercept."""
    try:
        x_arr = np.asarray(x, dtype=float)
        y_arr = np.asarray(y, dtype=float)
    except Exception:
        return {"ok": False, "error": "x and y must be numeric.", "result": None}

    if x_arr.ndim != 1 or y_arr.ndim != 1:
        return {"ok": False, "error": "x and y must be one-dimensional.", "result": None}

    if len(x_arr) != len(y_arr):
        return {"ok": False, "error": "x and y must have the same length.", "result": None}

    if len(x_arr) < 2:
        return {"ok": False, "error": "At least two data points are required.", "result": None}

    x_mean = x_arr.mean()
    y_mean = y_arr.mean()

    denominator = np.sum((x_arr - x_mean) ** 2)
    if denominator == 0:
        return {"ok": False, "error": "x values must not all be identical.", "result": None}

    slope = np.sum((x_arr - x_mean) * (y_arr - y_mean)) / denominator
    intercept = y_mean - slope * x_mean

    return {
        "ok": True,
        "error": None,
        "result": {
            "slope": slope,
            "intercept": intercept,
        }
    }


fit_result = fit_simple_linear_regression(train_df["study_hours"], train_df["quiz_score"])
fit_result

The output is a structured dictionary. When `ok=True`, the fitted model parameters appear inside `result`. The `slope` is the estimated change in quiz score for each additional study hour. The `intercept` is the fitted score when study hours are zero.

The intercept is a mathematical parameter, not necessarily a meaningful educational claim. In many models, parameter interpretation must be done carefully. A model can fit a line even when the extrapolated value at zero or beyond the data range is not reliable.

In [ ]:
def predict_simple_linear_regression(model: Dict[str, float], x: Any) -> Dict[str, Any]:
    """Make predictions using a fitted simple linear regression model."""
    if not isinstance(model, dict):
        return {"ok": False, "error": "model must be a dictionary.", "result": None}

    if "slope" not in model or "intercept" not in model:
        return {"ok": False, "error": "model must contain slope and intercept.", "result": None}

    try:
        x_arr = np.asarray(x, dtype=float)
    except Exception:
        return {"ok": False, "error": "x must be numeric.", "result": None}

    predictions = model["slope"] * x_arr + model["intercept"]

    return {
        "ok": True,
        "error": None,
        "result": predictions
    }


model = fit_result["result"]
train_predictions = predict_simple_linear_regression(model, train_df["study_hours"])
test_predictions = predict_simple_linear_regression(model, test_df["study_hours"])

print("Model:", model)
print("Test predictions:", test_predictions["result"])
print("Actual test labels:", test_df["quiz_score"].to_numpy())

The prediction output contains two predicted quiz scores for the held-out test rows. Compare these predictions with the actual test labels. They will not be identical because the model is a simplified line fitted from the training set. This difference between predicted and actual values is the prediction error.

In [ ]:
def mean_absolute_error(y_true: Any, y_pred: Any) -> Dict[str, Any]:
    """Compute mean absolute error with validation."""
    try:
        true_arr = np.asarray(y_true, dtype=float)
        pred_arr = np.asarray(y_pred, dtype=float)
    except Exception:
        return {"ok": False, "error": "y_true and y_pred must be numeric.", "result": None}

    if true_arr.shape != pred_arr.shape:
        return {"ok": False, "error": "y_true and y_pred must have the same shape.", "result": None}

    if true_arr.size == 0:
        return {"ok": False, "error": "inputs must not be empty.", "result": None}

    mae = np.mean(np.abs(true_arr - pred_arr))
    return {"ok": True, "error": None, "result": mae}


test_mae = mean_absolute_error(test_df["quiz_score"], test_predictions["result"])
test_mae

The MAE value is the average absolute prediction error on the test set. For example, an MAE of `3.0` would mean that, on average, the model's predictions are about three quiz-score points away from the actual values.

MAE is easy to interpret because it uses the same unit as the target variable. However, MAE alone is not enough to judge a model. You should also consider whether the data is representative, whether the feature is appropriate, and whether the model is being used in a high-impact context.

In [ ]:
x_line = np.linspace(data["study_hours"].min(), data["study_hours"].max(), 100)
line_predictions = predict_simple_linear_regression(model, x_line)["result"]

plt.figure(figsize=(7, 4))
plt.scatter(train_df["study_hours"], train_df["quiz_score"], label="Training data")
plt.scatter(test_df["study_hours"], test_df["quiz_score"], marker="x", s=80, label="Test data")
plt.plot(x_line, line_predictions, label="Fitted regression line")
plt.xlabel("Study hours")
plt.ylabel("Quiz score")
plt.title("Simple Linear Regression")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

The plot shows the training points, the held-out test points and the fitted regression line. This visualisation helps you see whether the model is broadly consistent with the data pattern. In later practicals, the features and models will be more complex, but visual and quantitative checks remain important.

<a id="m02a-testing"></a>

### 5. Testing and Analysis

We now test the modelling functions. The tests are not only checking mathematical output. They also check safe rejection of invalid inputs. This is consistent with the earlier M01 pattern: a workflow component should clearly report success or failure.

<div align="center">

<table>
<thead>
<tr><th><strong>Test type</strong></th><th><strong>Purpose</strong></th><th><strong>Example</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Normal</td><td>Check expected fitting and prediction behaviour.</td><td>Fit model on valid numeric data.</td></tr>
<tr><td align="left">Edge</td><td>Check minimal valid input.</td><td>Fit model with exactly two points.</td></tr>
<tr><td align="left">Failure</td><td>Reject invalid input safely.</td><td>Different lengths, identical x values, non-numeric values.</td></tr>
</tbody>
</table>

</div>

In [ ]:
normal_fit = fit_simple_linear_regression([1, 2, 3], [2, 4, 6])
assert normal_fit["ok"] is True
assert round(normal_fit["result"]["slope"], 4) == 2.0

edge_fit = fit_simple_linear_regression([1, 2], [3, 5])
assert edge_fit["ok"] is True
assert round(edge_fit["result"]["slope"], 4) == 2.0

bad_length = fit_simple_linear_regression([1, 2, 3], [1, 2])
assert bad_length["ok"] is False

bad_x = fit_simple_linear_regression([2, 2, 2], [1, 2, 3])
assert bad_x["ok"] is False

bad_type = fit_simple_linear_regression(["low", "high"], [1, 2])
assert bad_type["ok"] is False

bad_mae = mean_absolute_error([1, 2, 3], [1, 2])
assert bad_mae["ok"] is False

print("Regression pipeline tests passed.")

If this cell prints `Regression pipeline tests passed.`, the functions satisfy the behaviours we explicitly tested. As in earlier sessions, passing tests do not prove that the model is universally correct. They provide evidence that the implementation handles the specified normal, edge and failure cases.

<a id="m02a-student-tasks"></a>

### 6. Student Tasks

Extend the regression pipeline by adding a second small dataset and analysing the fitted model. Use only public or synthetic data. Do not use private student data or any personal information.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1</td><td>Create a small synthetic dataset with one numeric feature and one numeric label.</td><td>Practises defining features and labels.</td><td>A DataFrame with at least eight rows.</td></tr>
<tr><td align="left">Task 2</td><td>Fit a simple linear regression model.</td><td>Practises model fitting.</td><td>Structured output containing slope and intercept.</td></tr>
<tr><td align="left">Task 3</td><td>Make predictions and compute MAE.</td><td>Practises model evaluation.</td><td>Prediction array and MAE value.</td></tr>
<tr><td align="left">Task 4</td><td>Write a short interpretation of the model.</td><td>Connects numeric output to modelling meaning.</td><td>150 to 250 words.</td></tr>
</tbody>
</table>

</div>

Suggested dataset ideas include practice questions completed and quiz score, minutes spent reviewing unit notes and short-test score, number of retrieved document chunks and answer-quality score, or number of test cases passed and assignment score. These should be synthetic teaching examples unless you are using a clearly public dataset from [tulip-lab/open-data](https://github.com/tulip-lab/open-data).

In [ ]:
# Student task starter.
# Create your own small synthetic dataset here.

student_data = pd.DataFrame({
    "feature": [1, 2, 3, 4, 5, 6, 7, 8],
    "label":   [2, 3, 5, 7, 8, 10, 11, 13],
})

student_data

In [ ]:
# Student task starter.
# Fit, predict and evaluate your own dataset.

# TODO: split your data.
# student_train, student_test = train_test_split_simple(student_data, test_size=2)

# TODO: fit the model.
# student_fit = fit_simple_linear_regression(student_train["feature"], student_train["label"])

# TODO: make predictions.
# student_model = student_fit["result"]
# student_predictions = predict_simple_linear_regression(student_model, student_test["feature"])

# TODO: compute MAE.
# student_mae = mean_absolute_error(student_test["label"], student_predictions["result"])

# TODO: print or display your results.

<a id="m02a-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with the following evidence.

<div align="center">

<table>
<thead>
<tr><th><strong>Required item</strong></th><th><strong>What to submit</strong></th><th><strong>Quality check</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Dataset</td><td>A small public or synthetic dataset.</td><td>At least one feature column and one numeric label column.</td></tr>
<tr><td align="left">Model fitting</td><td>Output from <code>fit_simple_linear_regression</code>.</td><td><code>ok=True</code> and parameters are present.</td></tr>
<tr><td align="left">Predictions and MAE</td><td>Predicted values and error metric.</td><td>MAE is computed on held-out data.</td></tr>
<tr><td align="left">Tests</td><td>Evidence that the provided tests pass.</td><td>Normal, edge and failure behaviour is preserved.</td></tr>
<tr><td align="left">Reflection</td><td>150 to 250 words.</td><td>Reflection explains what the model learned and what its limitations are.</td></tr>
</tbody>
</table>

</div>

Reflection questions:

1. What are the feature and label in your dataset?
2. What does the slope mean in your example?
3. What does the MAE tell you about prediction error?
4. Why should a model be evaluated on held-out data?
5. How is this regression pipeline similar to later embedding, retrieval or evaluation pipelines?

#### Further Readings

- scikit-learn supervised learning tutorial: <https://scikit-learn.org/stable/supervised_learning.html>
- scikit-learn linear models: <https://scikit-learn.org/stable/modules/linear_model.html>
- pandas documentation: <https://pandas.pydata.org/docs/>
- NumPy documentation: <https://numpy.org/doc/>
- Public data repository for this unit: <https://github.com/tulip-lab/open-data>